In [ ]:
import monai
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
data_dir = 'data/MedNIST'
class_names = [d for d in os.listdir(data_dir) if os.path.isdir(f"{data_dir}/{d}")]
class_dirs = [f"{data_dir}/{d}" for d in class_names]

print(class_names)
print(class_dirs)
print(f"len(class_names): {len(class_names)}")

image_files = [
    sorted([f"{d}/{f}" for f in os.listdir(d)])
    for d in class_dirs
]

for ls in image_files:
    msg = f"length: {len(ls)},\t ex.{ls[0]}"
    print(msg)

In [ ]:
images = []
labels = []
for idx, ls in enumerate(image_files):
    images.extend(ls)
    t = [idx] * len(ls)
    labels.extend(t)
    
print(images[:3])
print(len(images))
print(labels[:3])
print(len(labels))

In [ ]:
N = len(images)
plt.subplots(3,3,figsize=(8,8))
for idx, k in enumerate(np.random.randint(N, size=9)):
    image = cv2.imread(images[k])
    label = labels[k]
    
    plt.subplot(3,3,idx+1)
    plt.imshow(image, cmap="gray", vmin=0, vmax=255)
    plt.xlabel(f"{class_names[label]}, {label}")

plt.tight_layout()
plt.show()

In [ ]:
train_len = int(N*0.8)
valid_len = int(N*0.1)
test_len = int(N*0.1)

datalist = [
    {
        "image":image,
        "label":label
    }for image, label in zip(images,labels)
]
np.random.shuffle(datalist)
print(datalist[:3])

idx = 0
train_datalist = datalist[idx:idx+train_len]

idx+=train_len
valid_datalist = datalist[idx:]

idx+=valid_len
test_datalist = datalist[idx:]

print(f"len(train_datalist): {len(train_datalist)}")
print(f"len(valid_datalist): {len(valid_datalist)}")
print(f"len(test_datalist): {len(test_datalist)}")

In [ ]:
from monai.transforms import(
    Compose,
    LoadImaged,
    AddChanneld,
    ScaleIntensityd,
    RandRotated,
    RandFlipd,
    RandZoomd,
    ToTensord,
    SelectItemsd,
    Activations,
    AsDiscrete
)
from monai.data import Dataset, DataLoader

In [ ]:
train_transforms = Compose(
  [
    LoadImaged(keys=["image"] ,image_only=True), 
    AddChanneld(keys=["image"]),                 
    ScaleIntensityd(keys=["image"]),
    RandRotated(
      keys=["image"], 
      range_x=np.pi / 12,                        
      prob=0.5, 
      keep_size=True
    ),
    RandFlipd(
      keys=["image"],
      spatial_axis=0,
      prob=0.5
    ),
    RandZoomd(
      keys=["image"],
      min_zoom=0.9, 
      max_zoom=1.1, 
      prob=0.5
    ),
    ToTensord(
      keys=["image", "label"]
    ),
    SelectItemsd(keys=["image", "label"])
  ]
)

valid_transforms = Compose(
  [
    LoadImaged(keys=["image"] ,image_only=True), 
    AddChanneld(keys=["image"]), 
    ScaleIntensityd(keys=["image"]), 
    ToTensord(keys=["image", "label"]),
    SelectItemsd(keys=["image", "label"])
  ]
)

num_class = 6
act = Activations(softmax=True)
to_onehot = AsDiscrete(to_onehot=num_class, n_classes=num_class)

In [ ]:
train_ds = Dataset(train_datalist, train_transforms)
valid_ds = Dataset(valid_datalist, valid_transforms)
test_ds = Dataset(test_datalist, valid_transforms)

train_loader = DataLoader(train_ds, batch_size=300, shuffle=True, num_workers=10)
valid_loader = DataLoader(valid_ds, batch_size=300, num_workers=10)
test_loader = DataLoader(test_ds, batch_size=300, num_workers=10)

In [ ]:
plt.subplots(3, 3, figsize=(8, 8))
for idx in range(9):
  img = train_ds[12]['image'][0]

  plt.subplot(3, 3, idx + 1)
  plt.imshow(img, cmap='gray')
plt.tight_layout()
plt.show()

In [ ]:
import torch
from monai.networks.nets import DenseNet121
from monai.metrics import compute_roc_auc

In [ ]:
device = torch.device("cuda:0")
model = DenseNet121(
  spatial_dims=2,
  in_channels=1,
  out_channels=num_class
).to(device)
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), 1e-5)
max_epochs = 4
val_interval = 1